# Thai Sentiment — PhayathaiBERT Fine-tuning with MLflow

This notebook fine-tunes the **PhayathaiBERT** transformer model on the preprocessed Wisesight sentiment splits, with training tracked and logged to MLflow.

## 1. Setup & Environment
Install dependencies, mount Google Drive, and set the working directory.

In [1]:
!pip install mlflow transformers datasets sentencepiece torch torchmetrics boto3 python-dotenv accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports

In [5]:
import os
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.pytorch
import torch
from dotenv import load_dotenv
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

## 3. Load Environment Variables
Loads `.env` (e.g. `HF_TOKEN`, `MLFLOW_TRACKING_URI`).

In [ ]:
load_dotenv('.env')

True

## 4. Logging Configuration

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 5. Configuration
Model checkpoint, label mapping, and training hyperparameters.

In [ ]:
# ── Config ──
MODEL_NAME    = "clicknext/phayathaibert"
LABEL_NAMES   = ["neg", "neu", "pos", "q"]
LABEL2ID      = {l: i for i, l in enumerate(LABEL_NAMES)}
ID2LABEL      = {i: l for i, l in enumerate(LABEL_NAMES)}
NUM_LABELS    = len(LABEL_NAMES)
MAX_LENGTH    = 128
BATCH_SIZE    = 8
EPOCHS        = 5
LEARNING_RATE = 2e-3
WARMUP_RATIO  = 0.1
WEIGHT_DECAY  = 0.01
OUTPUT_DIR    = "models/wangchanberta"

print("Config loaded ✅")

Config loaded ✅


## 6. Dataset Class
PyTorch `Dataset` wrapper that tokenizes each text sample on the fly.

In [ ]:
class ThaiSentimentDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = MAX_LENGTH):
        self.texts     = df["text_clean"].tolist()
        self.labels    = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        # ← no tokenizer() call here anymore

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(     # ← tokenize one sample at a time
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

## 7. Helper Functions

### 7.1 Trainer Metrics
Accuracy and F1 (weighted/macro) used by the HuggingFace `Trainer` during evaluation.

In [ ]:
# ── Metrics for Trainer ──
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1_w  = f1_score(labels, preds, average="weighted")
    f1_m  = f1_score(labels, preds, average="macro")
    return {
        "accuracy":    acc,
        "f1_weighted": f1_w,
        "f1_macro":    f1_m,
    }

### 7.2 Confusion Matrix Plotter

In [ ]:
# ── Confusion matrix helper ──
def plot_confusion_matrix(y_true, y_pred, title: str) -> plt.Figure:
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    plt.tight_layout()
    return fig

## 8. Load Data
Read the cleaned train/val/test CSVs produced by the preprocessing notebook.

In [ ]:
# ── Load data ──
train = pd.read_csv("data/processed/train.csv")
val   = pd.read_csv("data/processed/val.csv")
test  = pd.read_csv("data/processed/test.csv")

### 8.1 Split Sizes

In [ ]:
logger.info(f"Train : {len(train):,}")
logger.info(f"Val   : {len(val):,}")
logger.info(f"Test  : {len(test):,}")

## 9. Tokenizer & Tokenized Datasets
Load the PhayathaiBERT tokenizer and wrap each split in `ThaiSentimentDataset`.

In [ ]:
# ── Tokenizer ──
logger.info(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    token=os.getenv("HF_TOKEN"),
)

train_dataset = ThaiSentimentDataset(train, tokenizer)
val_dataset   = ThaiSentimentDataset(val,   tokenizer)
test_dataset  = ThaiSentimentDataset(test,  tokenizer)

logger.info("Datasets tokenized ✅")

## 10. Training Function
Fine-tunes PhayathaiBERT with the HuggingFace `Trainer`, logs params/metrics/artifacts to MLflow, evaluates on val/test, and registers the model.

In [ ]:
def train_phayathaibert(
    learning_rate: float = LEARNING_RATE,
    batch_size:    int   = BATCH_SIZE,
    epochs:        int   = EPOCHS,
    warmup_ratio:  float = WARMUP_RATIO,
    weight_decay:  float = WEIGHT_DECAY,
):
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
    mlflow.set_experiment("thai-sentiment / PhayathaiBERT")

    logger.info(f"Loading model: {MODEL_NAME}")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,   # ← suppress UNEXPECTED/MISSING warnings
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=learning_rate,
        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        eval_strategy="epoch",          # ← renamed from evaluation_strategy in transformers>=4.46
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    with mlflow.start_run(run_name="phayathaibert-finetune"):

        mlflow.log_params({
            "model_name":    MODEL_NAME,
            "max_length":    MAX_LENGTH,
            "learning_rate": learning_rate,
            "batch_size":    batch_size,
            "epochs":        epochs,
            "warmup_ratio":  warmup_ratio,
            "weight_decay":  weight_decay,
            "train_samples": len(train),
            "val_samples":   len(val),
            "test_samples":  len(test),
            "fp16":          torch.cuda.is_available(),
        })
        mlflow.set_tags({
            "method":   "PhayathaiBERT fine-tune",
            "language": "thai",
            "dataset":  "wisesight_sentiment",
        })

        logger.info("Starting training...")
        trainer.train()

        for log in trainer.state.log_history:
            step  = log.get("step", 0)
            epoch = log.get("epoch", None)
            if epoch is None:
                continue
            for key in ["eval_accuracy", "eval_f1_weighted", "eval_f1_macro",
                        "eval_loss", "loss"]:
                if key in log:
                    mlflow.log_metric(key, log[key], step=step)

        val_results = trainer.evaluate(val_dataset)
        mlflow.log_metrics({
            "val_accuracy":    val_results["eval_accuracy"],
            "val_f1_weighted": val_results["eval_f1_weighted"],
            "val_f1_macro":    val_results["eval_f1_macro"],
            "val_loss":        val_results["eval_loss"],
        })
        logger.info(f"Val → accuracy: {val_results['eval_accuracy']:.4f}  "
                    f"F1: {val_results['eval_f1_weighted']:.4f}")

        test_preds_output = trainer.predict(test_dataset)
        test_preds  = np.argmax(test_preds_output.predictions, axis=-1)
        test_labels = test["label"].tolist()   # ← already integers, no LABEL2ID mapping

        test_acc  = accuracy_score(test_labels, test_preds)
        test_f1_w = f1_score(test_labels, test_preds, average="weighted")
        test_f1_m = f1_score(test_labels, test_preds, average="macro")

        mlflow.log_metrics({
            "test_accuracy":    test_acc,
            "test_f1_weighted": test_f1_w,
            "test_f1_macro":    test_f1_m,
        })
        logger.info(f"Test → accuracy: {test_acc:.4f}  F1: {test_f1_w:.4f}")

        fig = plot_confusion_matrix(test_labels, test_preds, "PhayathaiBERT")
        mlflow.log_figure(fig, "confusion_matrix.png")
        plt.close(fig)

        report = classification_report(
            test_labels, test_preds, target_names=LABEL_NAMES
        )
        mlflow.log_text(report, "classification_report.txt")
        print(report)

        mlflow.pytorch.log_model(
            trainer.model,
            artifact_path="model",
            registered_model_name="thai-sentiment-phayathaibert",
        )

        tokenizer_dir = os.path.join(OUTPUT_DIR, "tokenizer")
        tokenizer.save_pretrained(tokenizer_dir)
        mlflow.log_artifacts(tokenizer_dir, artifact_path="tokenizer")

        run_id = mlflow.active_run().info.run_id
        logger.info(f"MLflow run ID: {run_id}")

    return trainer, {
        "test_accuracy":    test_acc,
        "test_f1_weighted": test_f1_w,
        "test_f1_macro":    test_f1_m,
    }

## 11. Run Training

In [ ]:
# ── Run training ──
trainer, results = train_phayathaibert(
    learning_rate=2e-3,
    batch_size=16,
    epochs=5,
)

print("\n=== Final Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")